In [1]:
import pandas as pd

In [2]:
df_normal = pd.read_json(r'..\data\민원(콜센터) 질의응답_다산콜센터_일반행정 문의_Training.json')
df_traffic = pd.read_json(r'..\data\민원(콜센터) 질의응답_다산콜센터_대중교통 안내_Training.json')

In [45]:
df = pd.concat([df_traffic, df_normal], axis = 0, ignore_index = True)

In [46]:
df['고객질문(요청)'].value_counts()

고객질문(요청)
                             58725
카드결제와 현금결제할 때 요금 차이가 있나요?       79
                                78
버스요금은 얼마인가요?                    55
시간은 얼마나 걸려요?                    54
                             ...  
꼭 서울시에 근무해야하나요?                  1
임대료는 어떻게 되나요?                    1
보증금도 있나요?                        1
입주순위도 있나요?                       1
1순위는 누가 되나요?                     1
Name: count, Length: 22824, dtype: int64

In [47]:
df = df.map(lambda x: str(x).strip())

In [48]:
df['고객질문(요청)'].value_counts()

고객질문(요청)
                             58807
카드결제와 현금결제할 때 요금 차이가 있나요?       79
시간은 얼마나 걸려요?                    55
버스요금은 얼마인가요?                    55
안녕하세요.                          46
                             ...  
꼭 서울시에 근무해야하나요?                  1
임대료는 어떻게 되나요?                    1
보증금도 있나요?                        1
입주순위도 있나요?                       1
1순위는 누가 되나요?                     1
Name: count, Length: 22721, dtype: int64

In [51]:
# 조건식: 현재 행에서 고객질문(요청) 데이터가 ''와 같지 않고 다음 행의 상담사답변이 ''와 같지 않은 경우
flag = (df['고객질문(요청)'] != '') & (df.shift(-1)['상담사답변'] != '')
# 조건식2: 현재 행에서 상담사 답변이 ''와 같지 않고 전 행의 고객 질문(요청) 데이터가 ''와 같지 않은 경우
flag2 = (df['상담사답변'] != '') & (df.shift(1)['고객질문(요청)'] != '')
df.loc[flag|flag2]

,도메인,카테고리,대화셋일련번호,화자,문장번호,고객의도,상담사의도,QA,고객질문(요청),상담사질문(요청),고객답변,상담사답변,개체명,용어사전,지식베이스
4,다산콜센터,대중교통 안내,B2033,고객,5,버스정류장,,Q,어느정류장에서 타야합니까?,,,,정류장,,정류장
5,다산콜센터,대중교통 안내,B2033,상담사,6,,버스정류장,A,,,,문성초등학교 정류장에서 탑승하시면 됩니다.,"문성초등학교, 정류장",문성초등학교/공공기관,"정류장,공공기관"
6,다산콜센터,대중교통 안내,B2033,고객,7,버스요금,,Q,버스요금은 얼마입니까?,,,,"버스, 요금",버스/교통수단/ 요금/돈,"요금,돈"
7,다산콜센터,대중교통 안내,B2033,상담사,8,,버스요금,A,,,,1200원 입니다.,1200원,1200원/금액,"1200원,금액"
8,다산콜센터,대중교통 안내,B2034,고객,1,버스노선,,Q,서울역에서 서울대학교가는 버스노선을 알고싶습니다.,,,,"서울, 역, 서울대학교, 버스, 노선",서울/지명/ 역/역사/ 서울대학교/지명/ 버스/교통수단,"역,교통수단"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89297,다산콜센터,일반행정 문의,B35809,상담사,16,,여성전용아파트,A,,,,"입주신청서와 추천서, 급여내역서 외 기타 서류를 갖고 홈페이지에서 신청하면 됩니다.","입주, 신청서, 추천서, 급여내역서, 서류, 홈페이지","입주/이사, 신청서, 추천서, 급여내역서, 서류/문서, 홈페이지/웹페이지/인터넷","신청서,인터넷"
89298,다산콜센터,일반행정 문의,B35809,고객,17,여성전용아파트,,Q,입주순위도 있나요?,,,,"입주, 순위","입주/이사, 순위/순서","순위,순서"
89299,다산콜센터,일반행정 문의,B35809,상담사,18,,여성전용아파트,A,,,,1~3순위가 있습니다.,순위,순위/순서,"순위,순서"
89300,다산콜센터,일반행정 문의,B35809,고객,19,여성전용아파트,,Q,1순위는 누가 되나요?,,,,순위,순위/순서,"순위,순서"


In [52]:
df2 = df.loc[flag,]
df2['상담사답변'] = df.loc[flag2, '상담사답변'].values

In [53]:
df3 = df.loc[flag,]
df3['상담사답변'] = df.shift(-1).loc[flag, '상담사답변'].values

In [54]:
# 파일로 저장

df3.to_csv('민원 질의응답(즉답형데이터).csv', index = False)

In [55]:
df3.to_excel("민원 질의응답(즉답형데이터).xlsx", index = False)

In [56]:
# 질문 중 중복 질문에 대한 제거

df3.drop_duplicates('고객질문(요청)', inplace = True)

In [57]:
from konlpy.tag import Komoran
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [58]:
komoran = Komoran()

def tokenize(text):
    return komoran.morphs(text)

vec = TfidfVectorizer(
    tokenizer = tokenize,
    lowercase = False,
    ngram_range = (1, 1),
    min_df = 5,
    max_df = 0.8
)

In [59]:
# 고객질문(요청) 데이터를 벡터화

X = vec.fit_transform(
    df3['고객질문(요청)']
)

c:\Users\hkssn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [60]:
new_questions = [
    '여권 재발급 신청 방법을 알려주세요',
    '전입 신고가 인터넷에서 가능한가요?',
    '지방세 환급금을 어디서 신청하나요?'
]

In [61]:
test = vec.transform(new_questions)

In [62]:
sims = cosine_similarity(test, X)
sims

array([[0.        , 0.        , 0.03768027, ..., 0.        , 0.        ,
        0.        ],
       [0.10382223, 0.        , 0.08867365, ..., 0.        , 0.        ,
        0.05229653],
       [0.02110115, 0.        , 0.03995857, ..., 0.03382659, 0.0321687 ,
        0.02175141]], shape=(3, 18948))

In [63]:
df3.reset_index(drop=True, inplace=True)

In [64]:
for idx, sim in enumerate(sims):
    question = new_questions[idx]

    sim_idxs = sim.argsort()[::-1]
    for i in sim_idxs[:2]:
        print(f"""
            고객의 질문: {question}
            유사 질문: {df3.loc[i, '고객질문(요청)']}
            유사도: {round(sim[i], 3)}
            답변: {df3.loc[i, '상담사답변']}
        """)


            고객의 질문: 여권 재발급 신청 방법을 알려주세요
            유사 질문: 신청방법을 알려주세요.
            유사도: 0.707
            답변: 주민등록상 세대주와 가까운 주민센터 또는 복지로 홈페이지에서 신청가능하세요.
        

            고객의 질문: 여권 재발급 신청 방법을 알려주세요
            유사 질문: 신청방법 좀 알려주세요?
            유사도: 0.655
            답변: 우선 사이트에 접속하셔서 회원가입을 해주세요. 청소년일경우 공인인증서가 없으면 본인확인절차를 거쳐 회원가입을 하고, 부모님이나 세대주분께서 가입을 하실 경우 공인인증서로 가입할 수 있습니다.
        

            고객의 질문: 전입 신고가 인터넷에서 가능한가요?
            유사 질문: 인터넷으로도 신고가능한가요?
            유사도: 0.62
            답변: 방문 접수밖에 안됩니다.
        

            고객의 질문: 전입 신고가 인터넷에서 가능한가요?
            유사 질문: 전입신고는 가서 해야되죠?
            유사도: 0.583
            답변: 방문신고는 신 거주지 동주민센터에서만 가능합니다.
        

            고객의 질문: 지방세 환급금을 어디서 신청하나요?
            유사 질문: 지방세 환급금 신청은 어떻게 해야하죠?
            유사도: 0.76
            답변: 인터넷에서 접수를 하셔야 합니다
        

            고객의 질문: 지방세 환급금을 어디서 신청하나요?
            유사 질문: 환급금을 기부할 수도 있나요?
            유사도: 0.566
            답변: 네 환급금을 사회복지공동모금회에 본인 명의로 기부가 가능합니다
        


- 고객 질문의 데이터를 이용해서 카테고리를 분류하는 모델 생성
    - 고객 질문 데이터를 이용하여 토큰화, 벡터화 (독립 변수)
    - 카테고리 일반 행정, 대중 교통 → 타겟 데이터
        - 카테고리 데이터를 LabelEncoder를 이용해서 수치화 변환
    - SVC 모델을 이용하여 벡터화된 데이터와 카테고리 데이터를 이용하여 학습
    - new_questions의 카테고리들을 확인
- 예측된 카테고리를 이용하여 df3에서 카테고리로 필터링
- new_questions를 벡터화하여 유사도를 확인

In [66]:
from sklearn.svm import SVC
from sklearn.preprocessing import LabelEncoder

In [67]:
X = df3['고객질문(요청)'].values
y = df3['카테고리'].values

In [68]:
X_vec = vec.fit_transform(X)

le = LabelEncoder()
y_le = le.fit_transform(y)

c:\Users\hkssn\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\sklearn\feature_extraction\text.py:526: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [70]:
svc = SVC(
    C = 1.0,
    kernel = 'linear',
    random_state = 42
)

In [71]:
svc.fit(X_vec, y_le)

,"C C: float, default=1.0Regularization parameter. The strength of the regularization isinversely proportional to C. Must be strictly positive. The penaltyis a squared l2 penalty. For an intuitive visualization of the effectsof scaling the regularization parameter C, see:ref:`sphx_glr_auto_examples_svm_plot_svm_scale_c.py`.",1.0
,"kernel kernel: {'linear', 'poly', 'rbf', 'sigmoid', 'precomputed'} or callable, default='rbf'Specifies the kernel type to be used in the algorithm. Ifnone is given, 'rbf' will be used. If a callable is given it is used topre-compute the kernel matrix from data matrices; that matrix should bean array of shape ``(n_samples, n_samples)``. For an intuitivevisualization of different kernel types see:ref:`sphx_glr_auto_examples_svm_plot_svm_kernels.py`.",'linear'
,"degree degree: int, default=3Degree of the polynomial kernel function ('poly').Must be non-negative. Ignored by all other kernels.",3
,"gamma gamma: {'scale', 'auto'} or float, default='scale'Kernel coefficient for 'rbf', 'poly' and 'sigmoid'.- if ``gamma='scale'`` (default) is passed then it uses 1 / (n_features * X.var()) as value of gamma,- if 'auto', uses 1 / n_features- if float, must be non-negative... versionchanged:: 0.22 The default value of ``gamma`` changed from 'auto' to 'scale'.",'scale'
,"coef0 coef0: float, default=0.0Independent term in kernel function.It is only significant in 'poly' and 'sigmoid'.",0.0
,"shrinking shrinking: bool, default=TrueWhether to use the shrinking heuristic.See the :ref:`User Guide `.",True
,"probability probability: bool, default=FalseWhether to enable probability estimates. This must be enabled priorto calling `fit`, will slow down that method as it internally uses5-fold cross-validation, and `predict_proba` may be inconsistent with`predict`. Read more in the :ref:`User Guide `.",False
,"tol tol: float, default=1e-3Tolerance for stopping criterion.",0.001
,"cache_size cache_size: float, default=200Specify the size of the kernel cache (in MB).",200
,"class_weight class_weight: dict or 'balanced', default=NoneSet the parameter C of class i to class_weight[i]*C forSVC. If not given, all classes are supposed to haveweight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.",None
,"verbose verbose: bool, default=FalseEnable verbose output. Note that this setting takes advantage of aper-process runtime setting in libsvm that, if enabled, may not workproperly in a multithreaded context.",False


In [72]:
new_vec = vec.transform(new_questions)
pred = svc.predict(new_vec)

In [73]:
pred

array([1, 1, 1])

In [74]:
# 라벨링된 데이터를 원본의 형태로 변환
le.inverse_transform(pred)

array(['일반행정 문의', '일반행정 문의', '일반행정 문의'], dtype=object)

In [76]:
new_questions = [
    '여권 재발급 신청 방법을 알려주세요',
    '전입 신고가 인터넷으로 가능한가요',
    '서울역에서 영등포로 가려면 어떻게 가나요',
    '강북구청 근처에서 가장 가까운 정류장은 어디인가요'
]

In [77]:
new_vec = vec.transform(new_questions)
pred2 = svc.predict(new_vec)

In [79]:
pred2_cate = le.inverse_transform(pred2)

In [80]:
for vec_data, cate in zip(new_vec, pred2_cate):
    x = vec.transform(
        df3.loc[ df3['카테고리'] == cate, '고객질문(요청)' ]
    )

    # 벡터화된 x와 vec_data를 기준으로 코사인 유사도를 계산: 질문이 1개이기 때문에 sims를 1차원으로 변환
    sims = cosine_similarity(vec_data, x).ravel()
    # 유사도를 내림차순 정렬로 인덱스의 값들을 확인
    idxs = sims.argsort()[::-1]
    for idx in idxs[:2]:
        print(f'유사도: {round(sims[idx], 3)}')
        print('유사 질문:', df3.loc[df3['카테고리'] == cate, '고객질문(요청)'].iloc[idx])
        print('답변:', df3.loc[df3['카테고리'] == cate, '상담사답변'].iloc[idx])

유사도: 0.707
유사 질문: 신청방법을 알려주세요.
답변: 주민등록상 세대주와 가까운 주민센터 또는 복지로 홈페이지에서 신청가능하세요.
유사도: 0.655
유사 질문: 신청방법 좀 알려주세요?
답변: 우선 사이트에 접속하셔서 회원가입을 해주세요. 청소년일경우 공인인증서가 없으면 본인확인절차를 거쳐 회원가입을 하고, 부모님이나 세대주분께서 가입을 하실 경우 공인인증서로 가입할 수 있습니다.
유사도: 0.584
유사 질문: 인터넷으로도 신고가능한가요?
답변: 방문 접수밖에 안됩니다.
유사도: 0.526
유사 질문: 인터넷으로 가능한가요?
답변: 인터넷으로 신청 가능합니다.
유사도: 0.671
유사 질문: 서울역에서 발산역 지하철로어떨게 가나요?
답변: 서울역에서 공항철도지하철을 이용하셔서 김포공항에 내리시고 5호선환승하셔서 발산역에서 내리시면됩니다
유사도: 0.571
유사 질문: 어떻게 가나요?
답변: 오류역에서 지하철을 탄 후 대전역 지하철에서 내리신 후 14번 버스를 타면됩니다.
유사도: 0.523
유사 질문: 그럼 가장 가까운 지하철역이 어디인가요?
답변: 신분당선 광교중앙역입니다.
유사도: 0.51
유사 질문: 정류장은 가깝나요?
답변: 도보로 5분 이동하시면 됩니다.
